In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [22]:
!pip install transformers datasets peft -q
!pip install -U torchao
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

OPTIONS  = ['A', 'B', 'C', 'D', 'E']
LABEL2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train shape: (2000, 8)
Test shape:  (500, 7)


In [6]:
# Q1 — Label Encoding

train['label'] = train['answer'].map(LABEL2ID)

q1_answer = train.iloc[150]['label']
print(f"\nQ1 — Encoded label for row index 150: {int(q1_answer)}")
print(f"Answer letter: {train.iloc[150]['answer']}")


Q1 — Encoded label for row index 150: 2
Answer letter: C


In [7]:
# Q2 — Prompt-Option Formatting

row0      = train.iloc[0]
prompt    = str(row0['prompt'])
option_b  = str(row0['B'])

formatted = prompt + " [SEP] " + option_b
q2_answer = len(formatted)

print(f"\nQ2 — Formatted string (row 0, Option B):")
print(f"'{formatted[:80]}...'")
print(f"Character length: {q2_answer}")


Q2 — Formatted string (row 0, Option B):
'Pick the best possible answer: What is Martin Heidegger's view on the relationsh...'
Character length: 407


In [23]:
# Q7 — LoRA Trainable Parameters

# Fresh model for LoRA
lora_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS
)

lora_model = get_peft_model(lora_base, lora_config)

total_params     = sum(p.numel() for p in lora_model.parameters())
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
q7_answer        = trainable_params

print(f"\nQ7 — LoRA Parameter Summary:")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"% trainable: {100 * trainable_params / total_params:.2f}%")
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco


Q7 — LoRA Parameter Summary:
Total parameters:     109,778,690
Trainable parameters: 295,681
% trainable: 0.27%
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [19]:
# Q8 — HuggingFace Dataset Preparation

def tokenize_row(row):
    formatted = [
        str(row['prompt']) + " [SEP] " + str(row[opt])
        for opt in OPTIONS
    ]
    tok = tokenizer(
        formatted,
        padding        = "max_length",
        truncation     = True,
        max_length     = 128,
    )
    return {
        'input_ids':      tok['input_ids'],       
        'attention_mask': tok['attention_mask'],   
        'labels':         LABEL2ID[row['answer']]
    }

# Build dataset from first 100 rows
data_list = [tokenize_row(train.iloc[i]) for i in range(100)]
hf_dataset = Dataset.from_list(data_list)
hf_dataset.set_format(type='torch')

first_item    = hf_dataset[0]
q8_answer     = len(first_item['input_ids'])  

print(f"\nQ8 — Dataset first item:")
print(f"input_ids shape: {first_item['input_ids'].shape}")
print(f"attention_mask shape: {first_item['attention_mask'].shape}")
print(f"label: {first_item['labels']}")
print(f"Number of choices in input_ids: {q8_answer}")


Q8 — Dataset first item:
input_ids shape: torch.Size([5, 128])
attention_mask shape: torch.Size([5, 128])
label: 1
Number of choices in input_ids: 5


In [26]:
# Q9 — Tiny LoRA Fine-Tuning 


# Build training dataset from first 32 rows
train32 = [tokenize_row(train.iloc[i]) for i in range(32)]

# Rebuild with max_length=64 for Q9
def tokenize_row_64(row):
    formatted = [
        str(row['prompt']) + " [SEP] " + str(row[opt])
        for opt in OPTIONS
    ]
    tok = tokenizer(
        formatted,
        padding        = "max_length",
        truncation     = True,
        max_length     = 64,
    )
    return {
        'input_ids':      tok['input_ids'],
        'attention_mask': tok['attention_mask'],
        'labels':         LABEL2ID[row['answer']]
    }

train32_64 = [tokenize_row_64(train.iloc[i]) for i in range(32)]
ds_train32 = Dataset.from_list(train32_64)
ds_train32.set_format(type='torch')

# Fresh LoRA model for training
lora_base2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_model2 = get_peft_model(lora_base2, LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS
))

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora-tiny",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps                   = 4,
    logging_steps               = 1,
    save_steps                  = 10,
    report_to                   = "none",
    fp16                        = torch.cuda.is_available(),
)

trainer = Trainer(
    model         = lora_model2,
    args          = training_args,
    train_dataset = ds_train32,
)

print("Starting tiny LoRA fine-tuning (max_steps=4)...")
trainer.train()

q9_answer = trainer.state.global_step
print(f"\nQ9 — Final global_step: {q9_answer}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting tiny LoRA fine-tuning (max_steps=4)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.605093
2,1.683490
3,1.681967
4,1.635499



Q9 — Final global_step: 4


In [27]:
# Q10 — Probability for Option E After Fine-Tuning

lora_model2.eval()

row0 = train.iloc[0]
formatted = [
    str(row0['prompt']) + " [SEP] " + str(row0[opt])
    for opt in OPTIONS
]

tok_q10 = tokenizer(
    formatted,
    padding        = "max_length",
    truncation     = True,
    max_length     = 64,
    return_tensors = "pt"
)

input_ids_q10      = tok_q10['input_ids'].unsqueeze(0)
attention_mask_q10 = tok_q10['attention_mask'].unsqueeze(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lora_model2.to(device)
input_ids_q10      = input_ids_q10.to(device)
attention_mask_q10 = attention_mask_q10.to(device)

with torch.no_grad():
    out     = lora_model2(input_ids=input_ids_q10, attention_mask=attention_mask_q10)
    logits  = out.logits  # [1, 5]
    probs   = torch.softmax(logits, dim=-1)[0]  # [5]

# Options: A=0, B=1, C=2, D=3, E=4
q10_answer = round(probs[4].item(), 4)

print(f"\nQ10 — Softmax probabilities after fine-tuning:")
for opt, prob in zip(OPTIONS, probs):
    print(f"Option {opt}: {prob.item():.4f}")
print(f"\nQ10 Answer — Probability of Option E: {q10_answer}")


Q10 — Softmax probabilities after fine-tuning:
Option A: 0.1979
Option B: 0.1931
Option C: 0.2012
Option D: 0.2031
Option E: 0.2047

Q10 Answer — Probability of Option E: 0.2047
